# La búsqueda del techo — el cuaderno que la corre

Este cuaderno **corre** la búsqueda de techos y nada más. No arma ninguna tabla ni conclusión — eso vive en `Benchmark_Search_Report_v1.ipynb`, que lee el registro que esto deja y lo presenta. Es la misma división que ya tienen la campaña y su informe, y por la misma razón: re-renderizar el informe cuesta segundos porque no vuelve a buscar nada.

Existía la mitad que dibuja y no la que corre: la única forma de pedir esta búsqueda era el paso `search-pilot` llamando a `harness.run_search()` por su cuenta, así que el artefacto que se abre y se lee nunca ejercitaba el programa que produce lo que muestra.

Se llamaba `Benchmark_Search_Pilot_v1`. El nombre afirmaba una escala que este cuaderno no tiene por qué elegir, y mientras la afirmaba **ningún cuaderno podía correr la búsqueda completa**: a escala completa los techos salían de la función de biblioteca y todo lo demás del recorrido salía de un cuaderno, que es la misma divergencia ---el ensayo ejercita una cosa y la corrida real hace otra--- que esta reestructura existe para cerrar, un nivel más abajo. `Benchmark_Search_v1` no se recicla: un nombre liberado que vuelve deja que una referencia vieja siga resolviendo, contra otro artefacto.

> **La escala la recibe, no la elige: es un modo del recorrido entero.**
>
> `config.is_pilot_scale()`, igual que la campaña, el barrido y el diagnóstico. En ensayo escribe `ceilings.pilot.json` y a escala completa `ceilings.json`, y en las dos el resto del recorrido lee el archivo de *su* escala: en ensayo todo corre y consume lo del ensayo, y a escala completa nada del ensayo se usa.
>
> **Acá estaba fijo en `True`.** El argumento escrito era que una sola respuesta elige las dos cosas ---a qué escala corre y a qué archivo escribe--- y que derivarla dejaría que el tamaño de la CAMPAÑA decidiera si este cuaderno lanza la corrida larga. La primera mitad es correcta y sigue siendo la razón por la que hay UNA respuesta y no dos. La segunda costaba más de lo que compraba: con `True` fijo, **ningún cuaderno podía correr la búsqueda a escala completa**, así que a escala completa los techos venían de la función de biblioteca mientras todo lo demás venía de un cuaderno. La autorización sigue existiendo y está donde corresponde ---en el paso `search-pilot`, que se niega cuando la escala configurada no es la del ensayo, exactamente como `campaign-local`, `noise-sweep` y `noise-diagnostic`---, así que abrir este cuaderno a mano a escala completa es una decisión que alguien toma; la forja no la toma sola.
>
> A escala de ensayo su respuesta no se cita. La rampa avanza con la fracción de entrenamiento transcurrida, así que a escala corta satura en la segunda época y todo techo se alcanza casi enseguida: lo que mediría es un paisaje en el que la campaña no entrena nunca. Lo que contesta es *«¿el programa corre?»*, que es lo único que un ensayo puede contestar --- y el registro lo dice de sí mismo con `atRequiredScale`, que es lo que hace que una campaña real se niegue a consumirlo.

In [ ]:
# Bootstrap: locate the repository wherever this is running, and import from it.
# Local, Colab and Kaggle differ only in where the checkout sits.
import os
import sys
from pathlib import Path


def find_repository() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in (os.environ.get("MIL_CREDA_REPO", ""), "/content", "/kaggle/working"):
        if base and Path(base).is_dir():
            candidates.append(Path(base))
            candidates.extend(sorted(Path(base).glob("*")))
    for candidate in candidates:
        if (candidate / "src" / "MIL_CREDA_Benchmark").is_dir():
            return candidate.resolve()
    raise SystemExit(
        "cannot find the repository. Set MIL_CREDA_REPO to the checkout that "
        "holds src/MIL_CREDA_Benchmark, or run this notebook from inside it."
    )


REPOSITORY = find_repository()
sys.path.insert(0, str(REPOSITORY / "src"))
print("repository:", REPOSITORY)

In [ ]:
from MIL_CREDA_Benchmark import config, harness

# La escala, recibida y con nombre. La misma respuesta elige las dos cosas --- a
# qué escala corre la búsqueda y a qué archivo escribe ---, así que escribirla en
# los dos lugares serían dos ortografías de una decisión, y la que quede vieja se
# lee igual de verde que la que no.
#
# `config.is_pilot_scale()` y no un `True` fijo: la escala es un modo del
# recorrido entero y no una elección de cada paso, y es la misma lectura que
# usan la campaña, el barrido y el diagnóstico --- las dos constantes que
# separan una escala de la otra son `EPOCHS` y `SEEDS`, leídas una sola vez y en
# un solo lugar. Con `True` fijo acá, ningún cuaderno podía correr la búsqueda
# completa, y a escala completa los techos salían de la biblioteca mientras todo
# el resto del recorrido salía de un cuaderno.
#
# La autorización para gastar la corrida larga no vive acá sino en el paso que
# ejecuta este cuaderno, que se niega cuando la escala configurada no es la del
# ensayo --- la misma guarda que ya tenían `campaign-local`, `noise-sweep` y
# `noise-diagnostic`, y por la misma razón: sus `produces` nombran el árbol de
# ensayo y ninguno más.
ES_ENSAYO = config.is_pilot_scale()
DESTINO = config.ceilings_record_for(ES_ENSAYO)

print("escala:", "ensayo" if ES_ENSAYO else "completa")
print("escribe en:", DESTINO)
print("registro en vigor ahora mismo:", config.ceilings_provenance()["source"])

## Lo que cuesta

El aforo declarado, y ningún pronóstico cronometrado. La campaña paga una corrida
real antes de comprometerse con la rejilla entera porque un estimado del costo es
más barato que el costo; acá el ensayo COMPLETO es del orden de esa corrida, así
que cronometrar una para pronosticarlo costaría una fracción visible de lo que
pronostica.

Se imprimen las dos: la escala declarada de la búsqueda y la que esta ejecución
va a correr. A escala completa son la misma, y verlas juntas es lo que dice cuál
de las dos está pasando antes de que empiece a gastar.

In [ ]:
aforo = config.search_sizing()
print(f"motor: {aforo['engine']}")
print(f"la búsqueda a su escala declarada: {aforo['trials']} trials de "
      f"{aforo['epochs']} épocas por (familia, transferencia) — "
      f"{aforo['runs']} corridas")
# Y lo que va a correr ESTA ejecución, que es lo de arriba a escala completa y
# la escala propia del ensayo si no. Las dos constantes salen de `config` y no
# de una aritmética escrita acá: el motor elige entre ellas con el mismo
# `pilot`, y un segundo cálculo sería la ortografía que queda vieja.
trials = config.PILOT_SEARCH_TRIALS if ES_ENSAYO else aforo["trials"]
epocas = config.PILOT_SEARCH_EPOCHS if ES_ENSAYO else aforo["epochs"]
print(f"esta corrida ({'ensayo' if ES_ENSAYO else 'completa'}): {trials} trials "
      f"de {epocas} épocas — "
      f"{trials * aforo['families'] * aforo['transfers']} corridas")

## La corrida

Se busca una vez. Si el registro ya existe se lee y no se vuelve a buscar: que el
registro exista significa que la búsqueda contestó, y sobrescribir una respuesta
porque alguien quería otra es el refinanciamiento silencioso que la negativa de la
campaña existe para impedir. Para volver a empezar hay que borrarlo a mano.

In [ ]:
# La biblioteca computa y este cuaderno orquesta: `run_search` arma la reducción
# a la escala que le corresponde a la búsqueda, despacha al motor que
# `config.SEARCH_ENGINE` declara y relee del disco lo que quedó escrito. Nada de
# eso se vuelve a escribir acá.
registro = harness.run_search(pilot=ES_ENSAYO)

print("registro escrito en:", DESTINO)
print("familias con techo:", ", ".join(sorted(registro)))
print()
print("las tablas y la conclusión las arma Benchmark_Search_Report_v1.ipynb")